In [ ]:
# Common imports + DES (Modelica) translator pieces.
import json
import shutil

from geojson_modelica_translator.system_parameters.system_parameters import (
    SystemParameters,
)

from lib.helpers import (
    DEFAULT_DOCKER_IMAGE_TAG,
    select_uo_class,
    setup_notebook_paths,
    copy_activity_to_new_location,
)

# -- Execution mode -----------------------------------------------------------
USE_DOCKER = False
UO = select_uo_class(USE_DOCKER, DEFAULT_DOCKER_IMAGE_TAG)

%load_ext autoreload
%autoreload 2

In [ ]:
# Standard set of working-directory paths used by every notebook.
# Change this to keep models/results outside the notebook folder (recommended).
# Examples: "../esbe_2026" (one level up) or "esbe_2026" (inside this folder).
MODEL_OUTPUT_SUBDIR = "../test_activities/esbe_2026"
paths = setup_notebook_paths(analysis_subdir=MODEL_OUTPUT_SUBDIR)
workdir = paths.workdir
analysis_dir = paths.analysis_dir
template_data_dir = paths.template_data_dir
num_usable_cores = paths.num_usable_cores

In [ ]:
# Copy activity_2/coincident to activity_4/coincident (deep copy of the full folder tree).
source_dir = paths.activity_dir("activity_2") / "coincident"
activity_4_dir = paths.activity_dir("activity_4")
dest_dir = activity_4_dir / "coincident"
copy_activity_to_new_location(source_dir, dest_dir)

# Set up the coincident project from activity_4/coincident
uo_coincident = UO(activity_4_dir, "coincident", template_dir=template_data_dir)

# Weather data is copied over from the activity_2/coincident project. No
# need to update the weather information.

In [ ]:
# Common paths used by the DES (Modelica) steps below.
# Saved as relative-style paths because they're consumed from within Docker.
scenario_path = uo_coincident.project_path / "baseline_scenario.csv"
feature_path = uo_coincident.project_path / "class_project_coincident.json"
sys_param_path = uo_coincident.project_path / "sys_params.json"

print(f"scenario_path: {scenario_path}")
print(f"feature_path: {feature_path}")
print(f"sys_param_path: {sys_param_path}")

# Rebuild sys params as a true 5G file.
sys_param_path_5g = uo_coincident.project_path / "sys_params_5g.json"

sp_5g = SystemParameters()
sp_5g.csv_to_sys_param(
    model_type="time_series",
    sys_param_filename=sys_param_path_5g,
    scenario_dir=uo_coincident.project_path / "run" / "baseline_scenario",
    feature_file=feature_path,
    # Use 5G_ghe because the CLI 5G creator currently requires GHE couplings.
    district_type="5G_ghe",
    # Absolute paths avoid cwd-dependent failures when creating 5G models.
    relative_path=None,
    modelica_load_filename="modelica.mos",
    overwrite=True,
)
print(f"Created sys params: {sys_param_path_5g}")

sys_param_data_5g = json.loads(sys_param_path_5g.read_text())
if "fifth_generation" not in sys_param_data_5g.get("district_system", {}):
    raise ValueError(
        f"Expected fifth_generation section in {sys_param_path_5g}, but it was not found."
    )

# Ensure loop-order input exists for 5G model creation.
loop_order_path = sys_param_path_5g.parent / "_loop_order.json"
fg = sys_param_data_5g.get("district_system", {}).get("fifth_generation", {})
loop_group = {
    "list_bldg_ids_in_group": [
        b["geojson_id"] for b in sys_param_data_5g.get("buildings", [])
    ]
}
borefields = fg.get("ghe_parameters", {}).get("borefields", [])
if borefields:
    loop_group["list_ghe_ids_in_group"] = [borefields[0]["ghe_id"]]
heat_sources = fg.get("heat_source_parameters", [])
if heat_sources:
    loop_group["list_source_ids_in_group"] = [heat_sources[0]["heat_source_id"]]
loop_order_path.write_text(json.dumps([loop_group], indent=2))
print(f"Wrote loop order file: {loop_order_path}")

des_scenario_name = "five_g_cli"
des_model_path = activity_4_dir / des_scenario_name

# Remove old output so we don't mistake stale files for a successful create.
if des_model_path.exists():
    shutil.rmtree(des_model_path)

log_file = uo_coincident.log_file
log_size_before = log_file.stat().st_size if log_file.exists() else 0
uo_coincident.des_create(
    sys_param_path_5g, feature_path, des_scenario_name, overwrite=True
)

# Detect backend failures that UOCliWrapper does not raise.
new_log_text = log_file.read_text()[log_size_before:] if log_file.exists() else ""
if "Traceback (most recent call last):" in new_log_text:
    raise RuntimeError(
        "uo des_create reported a Python traceback. Check coincident.log for details."
    )

print(f"Created 5G model at: {des_model_path}")

### 5G: CLI-based (this is not recommended)

In [ ]:
# Run the generated 5G DES model with UO CLI.
des_scenario_name = "five_g_cli"
des_model_path = activity_4_dir / des_scenario_name

if not (des_model_path / "package.mo").exists():
    raise FileNotFoundError(
        f"Expected generated 5G package not found: {des_model_path / 'package.mo'}"
    )

uo_coincident.des_run(
    des_model_path,
    start_time=5_097_600,  # March 1
    stop_time=6_307_200,  # March 14
    step_size=300,  # 5-minute step
)

results_path = des_model_path / f"{des_scenario_name}.Districts.district_results"
print(f"5G results path: {results_path}")